# TC3: 3x3 양자컴퓨터 실행 — PADO 방식 vs standard 방식 (parameter-shift)

케이스 하나(`configs/TC3.json`: 3x3 인스턴스 0, 상수 초기화 c = 0.1)에서 두 방식을 양자컴퓨터로 학습한다.

| 방식 | 시작점 | 양자컴퓨터 학습 |
|---|---|---|
| PADO (`pado_safe`) | PADO LWPP 사전학습 → 증류(\|θ\| ≤ 0.3 제거) 뒤의 θ, 증류된 회로 | parameter-shift 3 step (총 4 job, 370회로) |
| standard (`standard`) | 상수 초기값 θ_init, 전체 회로 | parameter-shift 12 step (총 13 job, 2,749회로) |

노트북 흐름: 설정 → (API 키) → backend → 1. 문제와 시작점 → 2. 회로와 대각 에너지표 → 3. 큐비트 배치와 transpile → 4. Session + Sampler 실행 (4-3 증류 전 측정 → 4-4 parameter-shift 본 학습) → 5. 요약 및 세션 종료

- 설정은 `configs/TC3.json`에서 읽는다. 시작점은 `prepare_start_points.py`가 GPU에서 미리 만든 `results_on_<GPU>/TC3/<케이스>/start_points/`.
- 결과는 `results_on_<backend 이름>/TC3/<케이스>/<방식>/`에 step마다 저장된다. 중간에 끊기면 학습 셀을 다시 실행하면 이어서 돌고, 끝난 방식은 건너뛴다.
- 지표·판정·그림은 이 노트북이 아니라 `python metric.py --exp TC3`가 만든다.

**실기기로 바꾸는 법**: `configs/TC3.json`의 `qpu.backend_mode`를 `"ibm"`, `qpu.ibm_backend`에 기기 이름을 넣는다. API 키는 아래 키 셀에 넣어 실행한 뒤 지운다. 비워 두면 이 컴퓨터에 저장된 IBM 계정(`~/.qiskit/qiskit-ibm.json`)을 쓴다.

## 설정 (`configs/TC3.json`)

In [1]:
import datetime
import importlib
import json
import time
import warnings
from pathlib import Path

import numpy as np
import rustworkx as rx
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2, Session

from common import device_tag, save_json, save_qasm   # quantum/common.py

REPO = Path.cwd().parent          # 이 노트북은 quantum/ 폴더에서 실행한다
cfg = json.loads((REPO / "configs" / "TC3.json").read_text())
q = cfg["qpu"]
BACKEND_MODE, FAKE_BACKEND, IBM_BACKEND = q["backend_mode"], q["fake_backend"], q["ibm_backend"]
SHOTS, STEPS, LR = q["shots"], q["steps"], cfg["lr"]                     # 회로당 shots, 방식별 step 수, Adam 학습률
RESILIENCE_LEVEL, REP_DELAY = q["resilience_level"], q["rep_delay"]       # 실기기 오차 완화 수준, shot 사이 대기(초, None = 기기 기본값)
OPT_LEVEL, SEED_TRANSPILER, TWO_Q_GATE = q["optimization_level"], q["seed_transpiler"], q["two_q_gate"]
CASE = f"{cfg['grid']}_inst{cfg['instance']}_seed{cfg['seed_base'] + cfg['instance']}"
START_DIR = REPO / f"results_on_{device_tag()}" / cfg["name"] / CASE / "start_points"   # 두 방식의 시작점 (GPU에서 준비)
print(f"{cfg['name']} {CASE} | {BACKEND_MODE} {IBM_BACKEND if BACKEND_MODE == 'ibm' else FAKE_BACKEND} | shots {SHOTS} | steps {STEPS}")

TC3 grid3x3_inst0_seed120490 | fake FakeNighthawk | shots 1024 | steps {'pado_safe': 5, 'standard': 20}


## API 키 (실기기에서만)

`qpu.backend_mode = "ibm"`일 때만 쓰인다. 키를 넣고 아래 backend 셀까지 실행한 뒤 이 셀의 키는 지운다. 키는 파일에 저장하지 않는다.

In [2]:
TOKEN = ""     # IBM Cloud API 키. 비우면 이 컴퓨터에 저장된 계정을 쓴다. 입력했으면 실행 뒤 지운다
INSTANCE = ""  # 인스턴스 CRN 또는 이름 (비우면 계정 기본값)

## backend

`FakeNighthawk`는 qiskit-ibm-runtime에 들어 있는 Nighthawk(정사각 격자 연결) 스냅샷이다. 패키지 스스로 "이 fake backend의 오류 값은 실제 기기의 일반적인 오류 값을 나타내도록 만든 것이 아니다"라고 경고하므로, fake 결과의 노이즈 크기는 실기기 예측으로 쓰지 않는다. 연결 구조와 transpile 결과, 코드 경로 검증용이다. 기기 정보(캘리브레이션 일자, rep_delay, 설정)는 `backend.json`에 저장한다.

In [3]:
warnings.filterwarnings("ignore", message="Properties of fake_")   # fake backend의 "오류 값이 실기기를 대표하지 않는다" 경고 (위 설명 참고)
if BACKEND_MODE == "ibm":
    if TOKEN:
        service = QiskitRuntimeService(channel="ibm_quantum_platform", token=TOKEN, instance=INSTANCE or None)
    else:
        service = QiskitRuntimeService()        # 이 컴퓨터에 저장된 계정
    backend = service.backend(IBM_BACKEND)
else:
    service = None
    backend = getattr(importlib.import_module("qiskit_ibm_runtime.fake_provider"), FAKE_BACKEND)()   # 기기 스냅샷(연결 구조 + 노이즈 모델)

cfg_b = backend.configuration()
rep = REP_DELAY if REP_DELAY is not None else cfg_b.default_rep_delay
if REP_DELAY is not None:
    assert cfg_b.rep_delay_range[0] <= REP_DELAY <= cfg_b.rep_delay_range[1], "rep_delay가 기기 허용 범위 밖"
try:
    calibration = str(backend.properties().last_update_date)
except Exception:
    calibration = None

OUT_ROOT = REPO / f"results_on_{backend.name}" / cfg["name"] / CASE   # 결과 폴더 이름에 기기 이름이 들어간다
OUT_ROOT.mkdir(parents=True, exist_ok=True)
backend_info = {"backend": backend.name, "mode": BACKEND_MODE, "num_qubits": backend.num_qubits,
                "processor_type": getattr(cfg_b, "processor_type", None), "calibration_date": calibration,
                "default_rep_delay_s": cfg_b.default_rep_delay, "rep_delay_range_s": list(cfg_b.rep_delay_range), "rep_delay_used_s": rep,
                "shots": SHOTS, "steps": STEPS, "resilience_level": RESILIENCE_LEVEL if BACKEND_MODE == "ibm" else None,
                "optimization_level": OPT_LEVEL, "seed_transpiler": SEED_TRANSPILER,
                "recorded_at": datetime.datetime.now().isoformat(timespec="seconds")}
save_json(OUT_ROOT / "backend.json", backend_info)
print(backend.name, backend.num_qubits, "qubits |", backend_info["processor_type"], "| 캘리브레이션", calibration)
print(f"rep_delay: 기본 {cfg_b.default_rep_delay * 1e6:.0f} µs, 허용 {cfg_b.rep_delay_range[0] * 1e6:.0f}~{cfg_b.rep_delay_range[1] * 1e6:.0f} µs, 이번 설정 {rep * 1e6:.0f} µs")
print("결과 폴더:", OUT_ROOT.relative_to(REPO))

fake_nighthawk 120 qubits | {'family': 'Nighthawk', 'revision': '1'} | 캘리브레이션 2025-12-08 09:37:09-05:00
rep_delay: 기본 250 µs, 허용 20~500 µs, 이번 설정 250 µs
결과 폴더: results_on_fake_nighthawk/TC3/grid3x3_inst0_seed120490


## 1. 문제와 시작점

In [4]:
# 시작점 파일 두 개: circuit.json(게이트 목록), params.json(θ 값, brute force E_min/E_max, 노이즈 없는 에너지)
circ_json = json.loads((START_DIR / "circuit.json").read_text())
start = json.loads((START_DIR / "params.json").read_text())
n, e_min, e_max = circ_json["n_qubits"], start["e_min"], start["e_max"]
ar = lambda e: (e_max - e) / (e_max - e_min)   # approximation ratio: E_max이면 0, 바닥 상태 E_min이면 1

# 방식별 회로와 시작 θ
#   standard : 전체 회로(각도 114개)와 상수 초기값
#   pado_safe: 증류 후 남은 게이트만 있는 회로(각도 61개)와 사전학습·증류를 거친 θ
gates = {"standard": circ_json["gates"], "pado_safe": circ_json["pruned_gates"]}
theta0 = {"standard": np.array(start["theta_init"]), "pado_safe": np.array(start["theta_pado_safe_start"])}
e_start = {"standard": start["exact_energy"]["standard_start"], "pado_safe": start["exact_energy"]["pado_safe_start"]}
for run in ["pado_safe", "standard"]:
    print(f"{run:10s} angles={len(theta0[run]):3d}  circuits/step={2 * len(theta0[run]) + 1:3d}  "
          f"noiseless start AR={ar(e_start[run]):.4f}")
print(f"brute force: E_min={e_min:.6f}, E_max={e_max:.6f}")

pado_safe  angles= 61  circuits/step=123  noiseless start AR=0.9311
standard   angles=114  circuits/step=229  noiseless start AR=0.5291
brute force: E_min=-8.656307, E_max=9.354969


## 2. 회로와 관측량

게이트 순서는 `circuit.json`과 같다. 각도 규약 R_P(θ) = exp(−iθP/2)는 Qiskit의 `rzz`, `rz`, `rx`와 같다. 파라미터는 `ParameterVector`로 만들어 행렬의 열 순서가 θ 번호와 같게 한다(이름이 `t10`, `t2`처럼 정렬되는 것을 피함). 마지막 줄은 노이즈 없는 Qiskit 에너지가 PennyLane으로 계산해 둔 시작점 에너지와 같은지 확인한다.

In [5]:
# 관측량 H = Σ J_ij Z_i Z_j + Σ h_i Z_i  (문제 해밀토니안; 전부 Z 기저 대각 항이므로 2^n개 비트열의 에너지를 미리 표로 만든다)
H = SparsePauliOp.from_sparse_list([("ZZ", [u, v], w) for u, v, w in circ_json["edges"]] +
                                   [("Z", [q], h) for q, h in circ_json["fields"]], num_qubits=n)
_idx = np.arange(1 << n, dtype=np.uint16)
_sz = np.array([1.0 - 2.0 * ((_idx >> q) & 1) for q in range(n)])   # 상태 인덱스 비트 q = (idx >> q) & 1: |0>→+1, |1>→−1
H_diag = np.zeros(1 << n)
for u, v, w in circ_json["edges"]:
    H_diag += w * _sz[u] * _sz[v]
for q, h in circ_json["fields"]:
    H_diag += h * _sz[q]
def sampler_evs_stds(pub_res):                                      # SamplerV2 결과(BitArray) → 행별 평균 에너지와 표준오차
    ba = pub_res.data.meas
    ints = (ba.array[..., 0].astype(np.uint16) << 8) | ba.array[..., 1].astype(np.uint16)
    e_shots = H_diag[ints]
    return e_shots.mean(axis=-1), e_shots.std(axis=-1, ddof=1) / np.sqrt(ba.num_shots)

# 방식별 파라미터 회로: circuit.json의 게이트를 순서대로 Qiskit 게이트로 옮긴다
circuits = {}
for run in ["pado_safe", "standard"]:
    theta = ParameterVector("theta", len(theta0[run]))   # theta[0], theta[1], ... 번호 순서가 θ 배열 순서와 같다
    qc = QuantumCircuit(n)
    for g in gates[run]:
        if g["gate"] == "H":
            qc.h(g["wires"][0])
        elif g["gate"] == "RZZ":                         # exp(-i θ ZZ / 2)
            qc.rzz(theta[g["param"]], *g["wires"])
        elif g["gate"] == "RZ":                          # exp(-i θ Z / 2)
            qc.rz(theta[g["param"]], g["wires"][0])
        elif g["gate"] == "RX":                          # exp(-i θ X / 2)
            qc.rx(theta[g["param"]], g["wires"][0])
    circuits[run] = qc
    # 확인: 노이즈 없는 Qiskit 에너지 == 시작점 파일의 에너지(PennyLane으로 계산)
    e = float(StatevectorEstimator().run([(qc, H, theta0[run])]).result()[0].data.evs)
    assert abs(e - e_start[run]) < 1e-8, (run, e, e_start[run])
    print(f"{run:10s} noiseless E(theta0) = {e:+.10f}  (start_points: {e_start[run]:+.10f})")

pado_safe  noiseless E(theta0) = -7.4149925623  (start_points: -7.4149925623)
standard   noiseless E(theta0) = -0.1747624990  (start_points: -0.1747624990)


## 3. 큐비트 배치와 transpile

### 3-1. 큐비트 배치: 3x3 격자
transpile에 배치를 맡기면 논리 큐비트 9개가 기기의 아무 물리 큐비트에나 놓일 수 있다. 그러면 격자에서 이웃인 두 큐비트가 기기에서는 멀리 떨어져 SWAP이 늘어날 수 있다. 그래서 문제의 3x3 격자(최근접 이웃 연결 12개)가 기기의 연결 구조 안에 **그대로** 들어가는 물리 큐비트 배치를 직접 고른다.

1. 논리 격자 그래프: 큐비트 q = 3·i + j (i행 j열), 오른쪽·아래쪽 이웃과 연결 (12개).
2. 기기의 연결 그래프 안에서 이 격자가 부분 그래프로 들어가는 모든 배치를 찾는다 (`rustworkx.vf2_mapping`).
3. 배치마다 점수 = 격자 연결 12쌍의 2큐비트 게이트 오류 합 + 9큐비트의 측정 오류 합. 가장 작은 배치를 쓴다 (오류 값은 기기의 최신 보정 데이터 `backend.target`).
4. 두 방식 모두 같은 배치를 쓴다.

대각선(NNN) 연결 8개는 정사각 격자 기기에서도 바로 연결돼 있지 않으므로 transpile이 SWAP을 넣는다. heavy-hex 기기(예: FakeFez)에는 3x3 격자가 그대로 들어가는 배치가 없다. 그 경우에는 그렇게 출력하고 transpiler의 자동 배치를 쓴다.

In [6]:
# 1. 논리 3x3 격자의 최근접 이웃 연결
grid = rx.PyGraph()
grid.add_nodes_from(range(n))
for i in range(3):
    for j in range(3):
        if j + 1 < 3:
            grid.add_edge(3 * i + j, 3 * i + j + 1, None)       # 오른쪽 이웃
        if i + 1 < 3:
            grid.add_edge(3 * i + j, 3 * (i + 1) + j, None)     # 아래쪽 이웃

# 2. 기기 연결 그래프 안에 격자가 그대로 들어가는 모든 배치, 3. 오류 점수
coupling = backend.coupling_map.graph.to_undirected(multigraph=False)
twoq_err = backend.target[TWO_Q_GATE]
candidates = []
for mapping in rx.vf2_mapping(coupling, grid, subgraph=True, induced=False):   # {물리 큐비트: 논리 큐비트}
    layout = [0] * n
    for phys, logical in mapping.items():
        layout[logical] = phys                                  # layout[논리 큐비트] = 물리 큐비트
    score = 0.0
    for u, v in grid.edge_list():
        props = twoq_err.get((layout[u], layout[v])) or twoq_err.get((layout[v], layout[u]))
        score += props.error if props is not None and props.error is not None else 1.0
    for qb in range(n):
        props = backend.target["measure"].get((layout[qb],))
        score += props.error if props is not None and props.error is not None else 1.0
    candidates.append((score, layout))

if candidates:
    score, LAYOUT = min(candidates)
    print(f"3x3 격자 배치 {len(candidates)}개 중 오류 점수가 가장 작은 배치 (점수 {score:.4f})")
    print("물리 큐비트 번호를 논리 격자 모양으로 (행 i, 열 j):")
    for i in range(3):
        print("   " + "  ".join(f"{LAYOUT[3 * i + j]:4d}" for j in range(3)))
else:
    LAYOUT = None
    print(f"{backend.name}에는 3x3 격자가 그대로 들어가는 배치가 없다 (heavy-hex 등). transpiler 자동 배치를 쓴다.")

3x3 격자 배치 640개 중 오류 점수가 가장 작은 배치 (점수 0.0682)
물리 큐비트 번호를 논리 격자 모양으로 (행 i, 열 j):
     27    28    29
     37    38    39
     47    48    49


### 3-2. transpile

양자컴퓨터는 정해진 기본 게이트(Nighthawk는 CZ, RZ, SX, X)와 연결된 큐비트 쌍에서만 2큐비트 게이트를 실행할 수 있다. transpile은 위 회로를 그 조건에 맞는 회로(ISA 회로)로 바꾼다. `initial_layout=LAYOUT`으로 3-1에서 고른 배치를 고정하고, 연결되지 않은 쌍(대각선 NNN)만 SWAP으로 잇는다. 회로 끝에 Z 기저 측정(`measure_all`)을 붙여 한 번만 transpile해 두고, 학습 중에는 파라미터 값만 바꿔 넣는다. 배치와 회로 크기는 `backend.json`에 기록한다.

In [7]:
pm = generate_preset_pass_manager(optimization_level=OPT_LEVEL, backend=backend, initial_layout=LAYOUT,
                                  seed_transpiler=SEED_TRANSPILER)
isa, phys = {}, {}
backend_info["layout_grid_fixed"] = LAYOUT is not None
backend_info["isa"] = {}
for run in ["pado_safe", "standard"]:
    qc_m = circuits[run].copy()
    qc_m.measure_all()                                    # SamplerV2용 Z 기저 측정 추가
    isa[run] = pm.run(qc_m)                               # 기기용 회로 (파라미터는 비어 있음)
    phys[run] = isa[run].layout.initial_index_layout(filter_ancillas=True)
    if LAYOUT is not None:
        assert phys[run] == LAYOUT, (run, phys[run], LAYOUT)   # 고른 격자 배치가 그대로 쓰였는지 확인
    two_q = sum(1 for inst in isa[run].data if inst.operation.num_qubits == 2)
    backend_info["isa"][run] = {"physical_qubits": phys[run], "two_qubit_gates": two_q, "depth": isa[run].depth(),
                                "two_qubit_depth": isa[run].depth(lambda inst: inst.operation.num_qubits == 2)}
    print(f"{run:10s} 2-qubit gates={two_q}  depth={isa[run].depth()}  2-qubit depth={backend_info['isa'][run]['two_qubit_depth']}  "
          f"physical qubits={phys[run]}")
save_json(OUT_ROOT / "backend.json", backend_info)

pado_safe  2-qubit gates=32  depth=73  2-qubit depth=15  physical qubits=[27, 28, 29, 37, 38, 39, 47, 48, 49]
standard   2-qubit gates=156  depth=309  2-qubit depth=83  physical qubits=[27, 28, 29, 37, 38, 39, 47, 48, 49]


## 4. parameter-shift 학습

### parameter-shift란
이 회로의 게이트는 모두 R_P(θ) = exp(−iθP/2) 꼴이다(P는 Z⊗Z, Z, X). 이런 게이트 하나의 각도 θ_i만 바꾸면 에너지 E는 θ_i에 대한 사인 곡선이 된다. 그래서 θ_i 방향의 기울기를 근사 없이 두 번의 측정으로 구할 수 있다.

  **g_i = ∂E/∂θ_i = ( E(θ + π/2·e_i) − E(θ − π/2·e_i) ) / 2**

여기서 e_i는 i번째 성분만 1인 벡터다(번호는 코드와 같이 0부터 센다). 즉 파라미터마다 "θ_i만 +π/2 옮긴 회로"와 "θ_i만 −π/2 옮긴 회로"를 한 번씩 돌려, 두 에너지의 차를 2로 나눈다.

### job 하나 = step 하나 (`SamplerV2` + `Session` 모드)
한 step에 필요한 회로를 모두 한 job에 담아 보낸다. 실제로 보내는 것은 파라미터가 비어 있는 측정 회로 1개(`isa[run]`)와 아래 파라미터 행렬(`rows`)을 묶은 PUB(primitive unified bloc) 1개다.

```
sampler.run([ (ISA 측정 회로, rows) ], shots=SHOTS)   ← job 1개

rows (2N+1 행 × N 열), N = 파라미터 수 (PADO 61, standard 114), 파라미터 번호 i = 0 .. N−1
  행 0      : θ                     → E(θ)       이 step의 에너지 (AR 기록용)
  행 1      : θ + π/2·e_0           → E_0+
  행 2      : θ − π/2·e_0           → E_0−
  ...
  행 2i+1   : θ + π/2·e_i           → E_i+
  행 2i+2   : θ − π/2·e_i           → E_i−
  ...
  행 2N     : θ − π/2·e_(N−1)       → E_(N−1)−

비트열 표본 → 대각 에너지 표(H_diag)로 E_r 계산 → 기울기 g_i = (E_i+ − E_i−) / 2 → Adam 업데이트 → 다음 step의 job
```

해밀토니안의 항(ZZ, Z)은 모두 Z 기저에서 대각화되어 있으므로, `SamplerV2`로 행마다 회로를 `SHOTS`번 측정해 얻은 비트열 표본을 `H_diag`에 대입하면 모든 항의 합산 기댓값과 표준오차를 한 번에 얻는다(별도 오차 완화 `resilience_level` 미적용). 마지막 업데이트 뒤에는 최종 θ의 에너지만 재는 회로 1개짜리 job을 하나 더 보낸다.

- **Session 모드 연속 실행**: 실기기(`BACKEND_MODE == "ibm"`)에서는 **4-1 셀**에서 `Session(backend=backend)`을 열어 두고 **4-3 셀**(증류 전 측정 1개 job)과 **4-4 셀**(본 학습 루프 17개 job)을 연달아 실행(`Shift+Enter` 연속 입력)한다. **가장 첫 번째 job(`pre_distill`, 회로 1개)만 일반 대기열(QUEUED)을 기다리고, 이 첫 job이 실행(RUNNING)되는 순간 세션이 활성화되어 뒤이은 본 학습 17개 job은 대기열 없이 즉시 연속 실행**된다. 모든 측정이 끝나면 **5번 셀**에서 `session.close()`로 세션을 닫는다.

| 방식 | step 수 | job당 회로 | 총 job | 총 회로 (512 shots 기준 총 샷) |
|---|---|---|---|---|
| PADO (`pado_safe`) | 3 | 2·61 + 1 = 123 | 3 + 1 = 4 | 3·123 + 1 = 370 (189,440 shots) |
| standard (`standard`) | 12 | 2·114 + 1 = 229 | 12 + 1 = 13 | 12·229 + 1 = 2,749 (1,407,488 shots) |

매 step의 θ는 노이즈 없는 statevector로도 평가해(`energy_noiseless`) 하드웨어 노이즈와 최적화 진행을 나눠 볼 수 있게 한다.

### 4-1. Sampler 및 Session 설정

In [8]:
session = Session(backend=backend) if BACKEND_MODE == "ibm" else None   # 실기기: 첫 job 이후 대기열 없이 연속 실행되는 Session 모드
sampler = SamplerV2(mode=session if session is not None else backend)
sampler.options.default_shots = SHOTS        # 회로(행) 하나당 측정 횟수
if BACKEND_MODE == "ibm" and REP_DELAY is not None:
    sampler.options.execution.rep_delay = REP_DELAY   # shot 사이 대기 (기기 허용 범위 안)
if session is not None:
    det = session.details() or {}
    print(f"Session 생성 완료: id={session.session_id} | max_time={det.get('max_time')}s | interactive_timeout={det.get('interactive_timeout')}s")

### 4-2. job 하나의 구성 보기

PADO 방식의 첫 step에서 job 하나에 들어갈 파라미터 행렬을 만들어 앞부분만 출력해 본다. 아무것도 보내지 않는다. 4-3의 학습 루프도 같은 방식으로 행렬을 만든다.

In [9]:
theta = theta0["pado_safe"]
k = len(theta)
rows = [theta.copy()]                      # 행 0: θ 그대로
for i in range(k):
    plus = theta.copy()
    plus[i] += np.pi / 2                   # 행 2i+1: θ_i만 +π/2
    minus = theta.copy()
    minus[i] -= np.pi / 2                  # 행 2i+2: θ_i만 −π/2
    rows += [plus, minus]
rows = np.array(rows)

print(f"rows 크기 = {rows.shape}  (2N+1 = {2 * k + 1}개 회로, N = {k}개 파라미터)")
for r in range(5):
    changed = np.flatnonzero(rows[r] != theta)
    if len(changed) == 0:
        print(f"  행 {r}: θ 그대로                → E(θ)")
    else:
        i = changed[0]
        sign = "+" if rows[r][i] > theta[i] else "−"
        print(f"  행 {r}: θ_{i}만 {sign}π/2 ({theta[i]:+.4f} → {rows[r][i]:+.4f}) → E_{i}{sign}")
print("  ...")
print(f"이 행렬과 ISA 측정 회로 1개를 묶어 SamplerV2 job 1개로 보낸다. 행마다 {SHOTS} shots씩 실행된다.")

rows 크기 = (123, 61)  (2N+1 = 123개 회로, N = 61개 파라미터)
  행 0: θ 그대로                → E(θ)
  행 1: θ_0만 +π/2 (+0.3263 → +1.8971) → E_0+
  행 2: θ_0만 −π/2 (+0.3263 → -1.2445) → E_0−
  행 3: θ_1만 +π/2 (-1.3019 → +0.2689) → E_1+
  행 4: θ_1만 −π/2 (-1.3019 → -2.8727) → E_1−
  ...
이 행렬, ISA 회로 1개, 관측량 H를 묶어 job 1개로 보낸다. 행마다 1024 shots씩 실행된다.


### 4-3. PADO 방식의 증류 전 측정 (비교용 1회, 첫 Job으로 Session 활성화)

PADO 방식의 양자컴퓨터 학습은 증류된 회로에서 시작하므로, 사전학습 직후·증류 전 상태는 본 학습에서 재지 않는다. 증류가 AR을 얼마나 바꾸는지 보려고, LWPP 사전학습 직후의 θ(`theta_lwpp_final`, 각도 114개)를 증류 전 전체 회로(standard와 같은 ISA 회로)에 넣어 회로 1개짜리 job을 가장 먼저 한 번 보낸다. 이 측정은 비교용이라 PADO 방식의 양자 자원(회로 수)에는 넣지 않는다. 결과는 `pado_safe/pre_distill.json`에 저장하고, 이미 있으면 다시 재지 않는다.

- **실기기 실행 시 주의 (`interactive_timeout = 60초`)**: 이 셀(`4-3`)과 바로 아래 본 학습 셀(`4-4`)을 **연달아 실행(`Shift+Enter` 두 번)**해 두면, `pre_distill`이 끝난 직후 0.1초 안에 `4-4` 본 학습의 첫 job이 제출되어 세션 우선권이 끊기지 않고 유지된다.

In [ ]:
(OUT_ROOT / "pado_safe").mkdir(parents=True, exist_ok=True)
pre_path = OUT_ROOT / "pado_safe" / "pre_distill.json"
theta_pre = np.array(start["theta_lwpp_final"])          # LWPP 사전학습 직후, 증류 전 (전체 114개 각도)
if pre_path.exists():
    pre = json.loads(pre_path.read_text())
    print("pre_distill.json found, not measured again")
else:
    t0 = time.perf_counter()
    job = sampler.run([(isa["standard"], theta_pre[None, :])], shots=SHOTS)   # 전체 회로, 회로 1개
    current_job = job
    print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] [pre_distill (증류 전 비교용)] job 제출: {job.job_id()} (회로 1개 × {SHOTS} shots)", flush=True)
    if BACKEND_MODE == "ibm":
        last_st = None
        while True:
            st = job.status()
            st_name = st.name if hasattr(st, "name") else str(st)
            if st_name != last_st:
                print(f"  -> [{datetime.datetime.now().strftime('%H:%M:%S')}] [pre_distill] status={st_name} (경과 {time.perf_counter() - t0:.1f}s)", flush=True)
                last_st = st_name
            if st_name in ("DONE", "CANCELLED", "ERROR"):
                break
            time.sleep(10)
    res = job.result()[0]
    wall_s = time.perf_counter() - t0
    evs_pre, stds_pre = sampler_evs_stds(res)
    energy = float(evs_pre[0])
    usage_s, queue_s, exec_s = None, None, None
    try:
        usage_s = float(job.usage())
    except Exception:
        pass
    try:
        ts_raw = job.metrics().get("timestamps", {})
        to_dt = lambda x: x if hasattr(x, "timestamp") else datetime.datetime.fromisoformat(str(x).replace("Z", "+00:00"))
        if "created" in ts_raw and "running" in ts_raw:
            queue_s = (to_dt(ts_raw["running"]) - to_dt(ts_raw["created"])).total_seconds()
        if "running" in ts_raw and "finished" in ts_raw:
            exec_s = (to_dt(ts_raw["finished"]) - to_dt(ts_raw["running"])).total_seconds()
    except Exception:
        pass
    e_noiseless = float(StatevectorEstimator().run([(circuits["standard"], H, theta_pre)]).result()[0].data.evs)
    pre = {"what": "PADO after LWPP pre-training, before distillation (theta_lwpp_final on the full circuit)",
           "backend": backend.name, "shots": SHOTS, "job_id": job.job_id(), "theta": theta_pre.tolist(),
           "energy": energy, "energy_std": float(stds_pre[0]), "ar": ar(energy),
           "energy_noiseless": e_noiseless, "ar_noiseless": ar(e_noiseless), "n_circuits": 1,
           "wall_s": wall_s, "queue_s": queue_s, "exec_s": exec_s, "usage_s": usage_s}
    save_json(pre_path, pre)
# 확인: 노이즈 없는 값은 시작점 파일의 LWPP 직후 에너지(PennyLane)와 같아야 한다
assert abs(pre["energy_noiseless"] - start["exact_energy"]["lwpp_final"]) < 1e-8
print(f"  [완료 | pre_distill 끝 ({datetime.datetime.now().strftime('%H:%M:%S')})] measured AR={pre['ar']:.4f} (noiseless {pre['ar_noiseless']:.4f}) | usage={pre.get('usage_s')}s | job {pre['job_id']}", flush=True)

PADO before distillation: measured AR=0.8243  noiseless AR=0.9210  job 6dfe5615-f3b5-452a-a987-3bc723f2ce6e


### 4-4. 본 학습 루프 (`Session` 연속 실행 + 실시간 상태 출력)

**이 셀(`4-4`) 하나가 `pado_safe`(총 4 job) → `standard`(총 13 job) 순서로 총 17개의 양자컴퓨터 job을 연속 실행한다.**

| 순서 | 방식 (`run`) | 반복 구간 (`t`) | step당 제출 회로 수 | 하는 일 |
|---|---|---|---|---|
| **1** | `pado_safe` | `t = 0, 1, 2` (3회) | `2·61 + 1 = 123`개 | 측정 기울기로 Adam 파라미터 갱신 (`Session` 우선 실행 유지) |
| **2** | `pado_safe` | `t = 3` (최종 1회) | `1`개 | 3번 갱신이 끝난 최종 `θ`의 에너지·AR 측정 |
| **3** | `standard` | `t = 0 .. 11` (12회) | `2·114 + 1 = 229`개 | 상수 초기값에서 측정 기울기로 Adam 파라미터 갱신 (`Session` 우선 실행 유지) |
| **4** | `standard` | `t = 12` (최종 1회) | `1`개 | 12번 갱신이 끝난 최종 `θ`의 에너지·AR 측정 |

#### 셀 내부 8단계 동작 구조
- **[단계 0] 이어하기(Resume) 점검**: `params.json`이 이미 있으면 읽어서 완료된 방식·step은 건너뛰고, 멈춘 지점의 `θ`와 Adam 모멘트(`m, v`)부터 이어서 시작한다.
- **[단계 1] 파라미터 행렬(`rows`) 생성**: 학습 step(`t < STEPS`)이면 `(2k+1) × k` parameter-shift 행렬을, 마지막 확인(`t == STEPS`)이면 `1 × k` 행렬을 만든다.
- **[단계 2] `SamplerV2` Job 제출 및 ID 즉시 저장**: 결과를 기다리기 전에 `pending_job` ID를 파일에 먼저 기록해 연결이 끊겨도 재수집할 수 있게 한다.
- **[단계 3] 실시간 상태(`QUEUED` → `RUNNING` → `DONE`) 모니터링**: 실기기 대기 중 상태가 바뀔 때마다 경과 시간을 출력한다.
- **[단계 4] 비트열 후처리 및 시간 분리 집계**: 측정된 비트열 표본을 `H_diag`에 대입해 행별 평균 에너지(`evs`)·표준오차(`stds`)를 구하고, 큐 대기(`queue_s`)·순수 실행(`exec_s`)·과금(`usage_s`) 시간을 분리한다.
- **[단계 5] 무노이즈 기준값 계산**: 같은 `θ`를 상태벡터(`StatevectorEstimator`)로 평가해 노이즈 격차 `G(T)` 산출용 기준 에너지를 기록한다.
- **[단계 6] 기울기 계산 및 Adam 업데이트**: `g_i = (E_i+ − E_i−) / 2`로 기울기를 구하고 Adam 식(`lr=0.02`)으로 `θ`를 갱신한다.
- **[단계 7~8] Step별 파일 저장 및 QASM 내보내기**: 매 step 즉시 `params.json`을 덮어쓰고, 방식이 끝나면 시작·최종 회로를 OpenQASM 2.0으로 저장한다.

In [10]:
current_job = None   # 중단(Interrupt) 시 바로 아래 4-5 셀에서 상태 확인·취소할 수 있도록 현재 job 객체 보관
total_jobs = sum(STEPS[r] + 1 for r in ["pado_safe", "standard"])

for run in ["pado_safe", "standard"]:
    out = OUT_ROOT / run
    (out / "qasm").mkdir(parents=True, exist_ok=True)
    path = out / "params.json"

    # =========================================================================
    # [단계 0] 이어하기(Resume) 점검 및 초기 상태 로드
    # - 이미 최종 에너지(rec["final"])까지 측정된 방식은 건너뛴다.
    # - 중간에 멈춘 실행이 있으면 저장된 마지막 θ와 Adam 모멘트(m, v)를 그대로 복원한다.
    # =========================================================================
    if path.exists():
        rec = json.loads(path.read_text())
        if rec["final"] is not None:
            print(f"{run}: 끝난 실행이 있어 건너뜀 (결과는 덮어쓰지 않는다)", flush=True)
            continue
    else:
        rec = {"run": run, "status": "running", "experiment": cfg["name"], "problem": CASE, "backend": backend.name,
               "e_min": e_min, "e_max": e_max,
               "config": {"shots": SHOTS, "resilience_level": None,
                          "rep_delay_s": rep, "steps": STEPS[run], "lr": LR, "optimization_level": OPT_LEVEL,
                          "seed_transpiler": SEED_TRANSPILER, "n_params": len(theta0[run]),
                          "circuits_per_step": 2 * len(theta0[run]) + 1, "physical_qubits": phys[run]},
               "theta_start": theta0[run].tolist(), "theta": theta0[run].tolist(),
               "adam_m": [0.0] * len(theta0[run]), "adam_v": [0.0] * len(theta0[run]),
               "steps": [], "pending_job": None, "final": None}

    theta = np.array(rec["theta"])      # 현재 파라미터 벡터 (크기 k)
    m = np.array(rec["adam_m"])         # Adam 1차 모멘트 (기울기의 지수 이동 평균)
    v = np.array(rec["adam_v"])         # Adam 2차 모멘트 (기울기 제곱의 지수 이동 평균)
    k = len(theta)
    base_job_idx = 0 if run == "pado_safe" else (STEPS["pado_safe"] + 1)
    print(f"\n=== [{datetime.datetime.now().strftime('%H:%M:%S')}] [{run} 방식 시작] 총 {STEPS[run]} step + 최종 확인 1회 (step당 회로 {2 * k + 1}개 × {SHOTS} shots) ===", flush=True)

    # t = 0 .. STEPS-1: parameter-shift 측정(회로 2k+1개) + Adam 갱신
    # t = STEPS       : 모든 갱신이 끝난 최종 θ의 에너지만 확인(회로 1개)
    for t in range(len(rec["steps"]), STEPS[run] + 1):
        final = t == STEPS[run]
        job_idx = base_job_idx + t + 1
        stage = f"{run} 최종 에너지 확인 (t={t})" if final else f"{run} 학습 step {t + 1}/{STEPS[run]} (t={t})"

        # =====================================================================
        # [단계 1] 이번 Job에 묶어 보낼 파라미터 행렬(rows) 구성
        # - 학습 step(final=False): 행 0은 현재 θ, 행 2i+1은 +π/2, 행 2i+2는 −π/2 이동 → 총 (2k+1) × k
        # - 최종 확인(final=True) : 현재 θ 하나만 측정 → 총 1 × k
        # =====================================================================
        if final:
            rows = np.array([theta])
        else:
            rows = [theta.copy()]
            for i in range(k):
                plus = theta.copy()
                plus[i] += np.pi / 2
                minus = theta.copy()
                minus[i] -= np.pi / 2
                rows += [plus, minus]
            rows = np.array(rows)

        # =====================================================================
        # [단계 2] SamplerV2 Job 제출 및 pending_job ID 선(先)저장
        # - 만약 이전 실행에서 제출만 해두고 결과를 못 받은 job_id가 있으면 새로 제출하지 않고 재연결한다.
        # - Session 모드이므로 첫 번째 job 실행 이후에는 큐 대기 없이 바로 실행된다.
        # =====================================================================
        t0 = time.perf_counter()
        now_str = datetime.datetime.now().strftime("%H:%M:%S")
        if rec["pending_job"] is not None and BACKEND_MODE == "ibm":
            job = service.job(rec["pending_job"])
            print(f"[{now_str}] [전체 {job_idx:2d}/{total_jobs} | {stage}] 기존 pending job 재연결: {job.job_id()}", flush=True)
        else:
            job = sampler.run([(isa[run], rows)], shots=SHOTS)
            rec["pending_job"] = job.job_id()
            save_json(path, rec)
            print(f"[{now_str}] [전체 {job_idx:2d}/{total_jobs} | {stage}] job 제출: {job.job_id()} (회로 {len(rows)}개 × {SHOTS} shots)", flush=True)
        current_job = job

        # =====================================================================
        # [단계 3] 실기기 대기 중 상태 변화(QUEUED → RUNNING → DONE) 실시간 출력
        # - 10초 간격으로 상태를 확인하며, 상태가 바뀔 때마다 화면에 즉시 찍어 준다.
        # =====================================================================
        if BACKEND_MODE == "ibm":
            last_st = None
            while True:
                st = job.status()
                st_name = st.name if hasattr(st, "name") else str(st)
                if st_name != last_st:
                    print(f"  -> [{datetime.datetime.now().strftime('%H:%M:%S')}] [{stage}] status={st_name} (경과 {time.perf_counter() - t0:.1f}s)", flush=True)
                    last_st = st_name
                if st_name in ("DONE", "CANCELLED", "ERROR"):
                    break
                time.sleep(10)

        # =====================================================================
        # [단계 4] 결과 수신, 비트열 대각 에너지(H_diag) 후처리, 시간 분리 집계
        # - sampler_evs_stds(res): 측정된 비트열들을 H_diag에 대입해 행별 평균 에너지(evs)와 표준오차(stds) 산출
        # - timestamps에서 큐 대기 시간(queue_s: created→running)과 순수 실행 시간(exec_s: running→finished) 분리
        # =====================================================================
        res = job.result()[0]
        wall_s = time.perf_counter() - t0
        evs, stds = sampler_evs_stds(res)
        energy = float(evs[0])                       # 행 0 = 현재 θ의 측정 에너지
        usage_s, queue_s, exec_s, timestamps = None, None, None, {}
        try:
            usage_s = float(job.usage())             # IBM이 집계한 QPU 과금 시간(초)
        except Exception:
            pass
        try:
            ts_raw = job.metrics().get("timestamps", {})
            timestamps = {kk: str(vv) for kk, vv in ts_raw.items()}
            to_dt = lambda x: x if hasattr(x, "timestamp") else datetime.datetime.fromisoformat(str(x).replace("Z", "+00:00"))
            if "created" in ts_raw and "running" in ts_raw:
                queue_s = (to_dt(ts_raw["running"]) - to_dt(ts_raw["created"])).total_seconds()
            if "running" in ts_raw and "finished" in ts_raw:
                exec_s = (to_dt(ts_raw["finished"]) - to_dt(ts_raw["running"])).total_seconds()
        except Exception:
            pass

        # =====================================================================
        # [단계 5] 동일 θ의 무노이즈 상태벡터 에너지 계산 (노이즈 격차 G(T) 기록용, 학습에는 불사용)
        # =====================================================================
        e_noiseless = float(StatevectorEstimator().run([(circuits[run], H, theta)]).result()[0].data.evs)

        row = {"t": t, "job_id": job.job_id(), "theta": theta.tolist(), "energy": energy,
               "energy_std": float(stds[0]), "ar": ar(energy),
               "energy_noiseless": e_noiseless, "ar_noiseless": ar(e_noiseless), "n_circuits": len(rows),
               "wall_s": wall_s, "queue_s": queue_s, "exec_s": exec_s, "usage_s": usage_s, "timestamps": timestamps}

        # =====================================================================
        # [단계 6] Parameter-shift 기울기 계산 및 Adam 파라미터 업데이트
        # - 학습 step(final=False): g_i = (E_i+ − E_i−) / 2 계산 후 Adam으로 다음 step의 θ 갱신
        # - 최종 확인(final=True) : 업데이트 없이 rec["final"]에 최종 에너지 기록 후 완료 처리
        # =====================================================================
        if final:
            rec["final"] = row
            rec["status"] = "ok"
        else:
            grad = (evs[1::2] - evs[2::2]) / 2
            row["grad"] = grad.tolist()
            rec["steps"].append(row)

            step = t + 1
            m = 0.9 * m + 0.1 * grad
            v = 0.999 * v + 0.001 * grad ** 2
            m_hat = m / (1 - 0.9 ** step)            # 초반 편향 보정(bias correction)
            v_hat = v / (1 - 0.999 ** step)
            theta = theta - LR * m_hat / (np.sqrt(v_hat) + 1e-8)
            rec["theta"], rec["adam_m"], rec["adam_v"] = theta.tolist(), m.tolist(), v.tolist()

        # =====================================================================
        # [단계 7] Step 완료 결과 즉시 저장 및 단계 완료 로그 출력
        # =====================================================================
        rec["pending_job"] = None
        save_json(path, rec)
        qs = f"{queue_s:.1f}s" if queue_s is not None else "-"
        es = f"{exec_s:.1f}s" if exec_s is not None else "-"
        print(f"  [완료 {job_idx:2d}/{total_jobs} | {stage} 끝 ({datetime.datetime.now().strftime('%H:%M:%S')})] "
              f"AR={row['ar']:.4f} (noiseless {row['ar_noiseless']:.4f}) | {len(rows)}회로 | "
              f"벽시계 {wall_s:.1f}s (큐 {qs} + 실행 {es}) | usage {usage_s}s | job {row['job_id']}", flush=True)

    # =========================================================================
    # [단계 8] 해당 방식 완료 후 시작·최종 파라미터 회로를 OpenQASM 2.0 및 JSON으로 저장
    # =========================================================================
    save_qasm(out / "qasm" / f"{run}_start.qasm", n, gates[run], theta0[run])
    save_qasm(out / "qasm" / f"{run}_final.qasm", n, gates[run], theta)
    save_json(out / "circuit.json", {"n_qubits": n, "gates": gates[run], "edges": circ_json["edges"],
                                     "fields": circ_json["fields"], "convention": circ_json["convention"]})
    print(f"=== [{datetime.datetime.now().strftime('%H:%M:%S')}] [{run} 방식 전체 완료] "
          f"시작 AR {rec['steps'][0]['ar']:.4f} → 최종 AR {rec['final']['ar']:.4f} (저장: {out.relative_to(REPO)}) ===", flush=True)

In [ ]:
# [4-5] 상태 확인 및 비상 취소(Cancel) / Session 종료 셀 (필요할 때만 실행)
if "current_job" in globals() and current_job is not None:
    print("최근 job:", current_job.job_id(), "| status:", current_job.status())
if "session" in globals() and session is not None:
    print("session_id:", session.session_id, "| status:", session.status())

# ---- [비상용] 실행/대기 중인 job을 취소하고 세션을 닫으려면 아래 주석을 풀고 실행하세요:
# if current_job is not None: current_job.cancel(); print("job cancel 요청 완료:", current_job.job_id())
# if session is not None: session.close(); print("session close 완료")
# (특정 job_id 문자열로 직접 취소할 때: service.job("여기에_JOB_ID_입력").cancel())

## 5. 끝

여기까지가 양자컴퓨터 실행이다. 회로 수·shots 합계, AR 처음/끝, 판정과 그림은 `python metric.py --exp TC3`가 `results_on_<backend>/TC3/metrics/`에 만든다.

In [12]:
if session is not None:
    session.close()
    print("Session closed.")
total_usage = 0.0
for run in ["pado_safe", "standard"]:
    rec = json.loads((OUT_ROOT / run / "params.json").read_text())
    pts = rec["steps"] + ([rec["final"]] if rec["final"] is not None else [])
    u_run = sum(p.get("usage_s") or 0.0 for p in pts)
    total_usage += u_run
    print(f"{run:10s} step {len(rec['steps'])}/{STEPS[run]}  회로 {sum(p['n_circuits'] for p in pts):4d}  "
          f"AR {pts[0]['ar']:.4f} → {pts[-1]['ar']:.4f}  (noiseless {pts[0]['ar_noiseless']:.4f} → {pts[-1]['ar_noiseless']:.4f})  "
          f"usage={u_run:.1f}s")
if total_usage > 0:
    print(f"두 방식 합계 usage = {total_usage:.1f}s ({total_usage / 60:.1f}분, Pay-As-You-Go $1.60/s 기준 약 ${total_usage * 1.60:,.0f})")
if service is not None:
    try:
        u = service.usage()
        print("계정 usage(초):", {k: u.get(k) for k in ("usage_consumed_seconds", "usage_limit_seconds", "usage_remaining_seconds")})
    except Exception:
        pass
print(f"다음: python metric.py --exp TC3   (지표·판정·그림 → results_on_{backend.name}/TC3/metrics/)")

pado_safe  step 5/5  회로 616  AR 0.8912 → 0.9303  (noiseless 0.9311 → 0.9729)
standard   step 20/20  회로 4581  AR 0.5323 → 0.7478  (noiseless 0.5291 → 0.8183)
다음: python metric.py --exp TC3   (지표·판정·그림 → results_on_fake_nighthawk/TC3/metrics/)
